# Phase 2: AI Document Intelligence - PDF Integration

In this phase, I am upgrading the RAG system to handle external documents. Instead of using static text, the AI will now read and analyze information directly from PDF files.

### Learning Objectives:
* Extracting raw text from PDF documents.
* Breaking down large documents into smaller, readable "chunks".
* Storing document data into a searchable vector database (FAISS).

# Phase 2: AI Document Intelligence - PDF Integration

In this phase, I am upgrading the RAG system to handle external documents. Instead of using static text, the AI will now read and analyze information directly from PDF files.

### Learning Objectives:
* Extracting raw text from PDF documents.
* Breaking down large documents into smaller, readable "chunks".
* Storing document data into a searchable vector database (FAISS).

In [ ]:
!pip install -qU \
    langchain \
    langchain-groq \
    langchain-community \
    langchain-huggingface \
    sentence-transformers \
    faiss-cpu \
    pypdf

## 2. Setup and Imports
Setting up the environment and importing the tools needed for PDF processing.

In [ ]:
import os
from getpass import getpass
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Configure API Key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

## 3. Loading the PDF
I will use the `PyPDFLoader` to read the content of the uploaded document.

In [ ]:
# Make sure to upload your PDF to the directory first!
# For example, using 'SOP_Company.pdf'
file_path = "SOP_Company.pdf" 

loader = PyPDFLoader(file_path)
pages = loader.load()

print(f"Successfully loaded {len(pages)} pages from the document.")

## 4. Chunking the Text
Large documents are too big for the AI to read at once. We split the text into smaller pieces (chunks) so the AI can find answers more accurately.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(pages)
print(f"The document has been split into {len(docs)} smaller chunks.")

## 5. Creating the Vector Store
We transform the text chunks into mathematical vectors using HuggingFace and store them in FAISS.

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Build the vector database
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

## 6. Building the RAG Chain
Linking the document retriever with the Llama 3.3 model.

In [ ]:
# Simple professional prompt
prompt_template = """
Use the following context to answer the question. 
If you don't find the answer in the context, just say that you don't know.

Context:
{context}

Question: {question}

Answer:
"""
prompt = ChatPromptTemplate.from_template(prompt_template)

llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# The RAG pipeline
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## 7. Testing the System
Let's ask the AI a question based on the PDF content.

In [ ]:
query = "What is the penalty for being late according to the document?"
response = rag_chain.invoke(query)

print(f"Question: {query}")
print("-" * 20)
print(f"AI Response: {response}")